## Extracción de variables Landsat — v4

In [ ]:
# Instalación de dependencias del entorno Snowflake
!pip install uv
!uv pip install -r requirements.txt

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
# get_active_session() obtiene la conexión activa a Snowflake desde el entorno Jupyter
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
# pystac_client permite buscar imágenes en catálogos satelitales estándar (STAC)
import pystac_client
# planetary_computer es la interfaz de Microsoft para acceder a datos satelitales abiertos
import planetary_computer as pc
# stac_load descarga y organiza los datos de las imágenes en formato xarray
from odc.stac import stac_load
from tqdm import tqdm
import os

tqdm.pandas()
print('Imports OK')

In [ ]:
# Conectar al catálogo de Planetary Computer (Microsoft)
# Este catálogo indexa millones de imágenes satelitales accesibles vía API
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=pc.sign_inplace,  # firma automáticamente las URLs de descarga
)

# Tamaño del área de búsqueda alrededor de cada punto de muestra
# Landsat tiene resolución de 30 m/píxel, por lo que 100 m captura ~3x3 píxeles de agua
BBOX_WATER = 0.00089831  # ~100 m en grados decimales — zona de agua
BBOX_LAND  = 0.00449155  # ~500 m en grados decimales — zona de tierra circundante

eps = 1e-10  # valor pequeño para evitar divisiones por cero en los índices

print(f'Buffer agua: {BBOX_WATER}  (~100m)')
print(f'Buffer tierra: {BBOX_LAND}  (~500m)')

In [ ]:
def apply_qa_mask(data):
    """
    Elimina píxeles con nube, sombra de nube o nieve usando la banda QA (control de calidad).
    Cada bit de qa_pixel indica una condición: 1=con problema, 0=limpio.
    Solo se conservan píxeles donde los 4 bits relevantes son 0 (cielo despejado).
    """
    if 'qa_pixel' not in data:
        return data
    qa = data['qa_pixel'].astype('uint16')
    mask = (
        ((qa & (1 << 3)) == 0) &  # sin nube
        ((qa & (1 << 4)) == 0) &  # sin sombra de nube
        ((qa & (1 << 5)) == 0) &  # sin nieve
        ((qa & (1 << 1)) == 0)    # sin nube dilatada (borde de nube)
    )
    # Aplicar máscara: los píxeles con problema quedan como NaN
    for band in [b for b in data.data_vars if b != 'qa_pixel']:
        data[band] = data[band].where(mask)
    return data


def safe_med(arr):
    """Calcula la mediana de un array ignorando NaN. Retorna NaN si el resultado es 0."""
    v = float(arr.astype(float).median(skipna=True))
    return np.nan if (v == 0 or np.isnan(v)) else v


# Diccionario de valores nulos: se devuelve si no se encuentra imagen válida para una muestra
NAN_RESULT = {
    'nir': np.nan, 'green': np.nan, 'red': np.nan, 'blue': np.nan,
    'swir16': np.nan, 'swir22': np.nan,
    'NDMI': np.nan, 'MNDWI': np.nan, 'NDTI': np.nan,
    'NDVI_wat': np.nan, 'nir_blue': np.nan,
    'ndvi_land': np.nan, 'ndbi': np.nan, 'bsi': np.nan,
}

# Bandas a descargar de Landsat Collection 2 Level-2
BANDS = ['green', 'nir08', 'swir16', 'swir22', 'blue', 'red', 'qa_pixel']

print('Helpers OK')
print(f'Columnas de salida ({len(NAN_RESULT)}): {list(NAN_RESULT.keys())}')

In [ ]:
def compute_landsat_values_v4(row):
    """
    Para cada muestra del dataset extrae:
    - Bandas espectrales e índices del área de agua (buffer 100 m)
    - Índices de vegetación, urbanización y suelo del área terrestre (buffer 500 m)
    Busca la imagen más cercana en fecha con menos del 10% de nubosidad.
    """
    lat  = row['Latitude']
    lon  = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Definir los bounding boxes para agua y tierra
    hw = BBOX_WATER / 2
    hl = BBOX_LAND  / 2
    bbox_water = [lon - hw, lat - hw, lon + hw, lat + hw]
    bbox_land  = [lon - hl, lat - hl, lon + hl, lat + hl]

    try:
        # Buscar imágenes Landsat del período del reto con baja nubosidad
        search = catalog.search(
            collections=['landsat-c2-l2'],
            bbox=bbox_water,
            datetime='2011-01-01/2015-12-31',
            query={'eo:cloud_cover': {'lt': 10}},  # menos del 10% de nubes
        )
        items = search.item_collection()
        if not items:
            return pd.Series(NAN_RESULT)

        # Seleccionar la imagen más cercana en tiempo a la fecha de muestreo
        sample_date_utc = (
            date.tz_localize('UTC') if date.tzinfo is None else date.tz_convert('UTC')
        )
        items_sorted = sorted(
            items,
            key=lambda x: abs(
                pd.to_datetime(x.properties['datetime']).tz_convert('UTC') - sample_date_utc
            )
        )
        selected_item = pc.sign(items_sorted[0])  # firmar URL para descarga

        # ── Zona de agua (100 m) ──────────────────────────────
        dw = stac_load([selected_item], bands=BANDS, bbox=bbox_water).isel(time=0)
        dw = apply_qa_mask(dw)  # eliminar píxeles con nubes

        nir    = safe_med(dw['nir08'])
        green  = safe_med(dw['green'])
        swir16 = safe_med(dw['swir16'])
        swir22 = safe_med(dw['swir22'])
        blue   = safe_med(dw['blue'])
        red    = safe_med(dw['red'])

        # Función auxiliar para calcular índices normalizados (A-B)/(A+B)
        def idx(a, b):
            return (a - b) / (a + b + eps) if not (np.isnan(a) or np.isnan(b)) else np.nan

        # Índices espectrales del agua
        ndmi     = idx(nir, swir16)   # humedad de la vegetación
        mndwi    = idx(green, swir16) # detección de cuerpos de agua
        ndti     = idx(red, green)    # turbidez del agua
        ndvi_wat = idx(nir, red)      # vegetación en zona de agua
        nir_blue = nir / (blue + eps) if not np.isnan(blue) else np.nan

        # ── Zona de tierra circundante (500 m) ───────────────
        dl = stac_load([selected_item], bands=BANDS, bbox=bbox_land).isel(time=0)
        dl = apply_qa_mask(dl)

        nir_l    = dl['nir08'].astype(float)
        green_l  = dl['green'].astype(float)
        swir16_l = dl['swir16'].astype(float)
        blue_l   = dl['blue'].astype(float)
        red_l    = dl['red'].astype(float)

        # Excluir píxeles de agua del buffer de tierra usando MNDWI
        # MNDWI > 0 indica agua — solo se conserva tierra (MNDWI <= 0)
        mndwi_l   = (green_l - swir16_l) / (green_l + swir16_l + eps)
        land_mask = mndwi_l <= 0

        def lmed(arr):
            v = float(arr.where(land_mask).median(skipna=True))
            return np.nan if (np.isnan(v) or v == 0) else v

        nir_l2    = lmed(nir_l)
        red_l2    = lmed(red_l)
        swir16_l2 = lmed(swir16_l)
        blue_l2   = lmed(blue_l)

        ndvi_land = idx(nir_l2, red_l2)    # vegetación terrestre circundante
        ndbi      = idx(swir16_l2, nir_l2) # índice de suelo construido/urbanización

        # Índice de suelo desnudo — combina 4 bandas para detectar suelo sin vegetación
        if not any(np.isnan(x) for x in [swir16_l2, red_l2, nir_l2, blue_l2]):
            bsi = ((swir16_l2 + red_l2) - (nir_l2 + blue_l2)) /                   ((swir16_l2 + red_l2) + (nir_l2 + blue_l2) + eps)
        else:
            bsi = np.nan

        return pd.Series({
            'nir': nir, 'green': green, 'red': red, 'blue': blue,
            'swir16': swir16, 'swir22': swir22,
            'NDMI': ndmi, 'MNDWI': mndwi, 'NDTI': ndti,
            'NDVI_wat': ndvi_wat, 'nir_blue': nir_blue,
            'ndvi_land': ndvi_land, 'ndbi': ndbi, 'bsi': bsi,
        })

    except Exception:
        # Si falla cualquier paso (sin imagen, error de red, etc.), devolver NaN
        return pd.Series(NAN_RESULT)

## Extracción — entrenamiento

In [ ]:
# Cargar el dataset de calidad del agua con coordenadas y fechas de muestreo
water_quality_df = pd.read_csv('water_quality_training_dataset.csv')
print(f'Registros de entrenamiento: {len(water_quality_df)}')

In [ ]:
TRAIN_PARTIAL = 'landsat_v4_train_partial.csv'
TRAIN_FINAL   = 'landsat_features_training_v4.csv'

if os.path.exists(TRAIN_FINAL) and len(pd.read_csv(TRAIN_FINAL)) == len(water_quality_df):
    print(f'Ya completo ({len(water_quality_df)} filas), saltando')
else:
    # Retomar desde donde se dejó si hubo una interrupción anterior
    processed_rows = len(pd.read_csv(TRAIN_PARTIAL)) if os.path.exists(TRAIN_PARTIAL) else 0
    file_exists    = os.path.exists(TRAIN_PARTIAL)
    print(f'Reanudando desde fila {processed_rows}')

    # Procesar en lotes de 50 para guardar avance periódicamente
    for start in range(processed_rows, len(water_quality_df), 50):
        end   = min(start + 50, len(water_quality_df))
        batch = water_quality_df.iloc[start:end]
        print(f'Filas {start}–{end}')

        # Aplicar la función de extracción fila por fila
        feats = batch.progress_apply(compute_landsat_values_v4, axis=1)
        feats['Latitude']    = batch['Latitude'].values
        feats['Longitude']   = batch['Longitude'].values
        feats['Sample Date'] = batch['Sample Date'].values

        # Añadir al CSV parcial (mode='a' = append, no sobreescribir)
        feats.to_csv(TRAIN_PARTIAL, mode='a', header=not file_exists, index=False)
        file_exists = True

    # Consolidar en CSV final con columnas en orden definido
    df = pd.read_csv(TRAIN_PARTIAL)
    col_order = ['Latitude', 'Longitude', 'Sample Date',
                 'nir', 'green', 'red', 'blue', 'swir16', 'swir22',
                 'NDMI', 'MNDWI', 'NDTI', 'NDVI_wat', 'nir_blue',
                 'ndvi_land', 'ndbi', 'bsi']
    df = df[[c for c in col_order if c in df.columns]]
    df.to_csv(TRAIN_FINAL, index=False)
    if os.path.exists(TRAIN_PARTIAL):
        os.remove(TRAIN_PARTIAL)
    print(f'Extraccion completa: {len(df)} registros')

train_df = pd.read_csv(TRAIN_FINAL)
# Verificar que el número de filas coincide con el dataset original
assert len(train_df) == len(water_quality_df), f'Desfase: {len(train_df)} vs {len(water_quality_df)}'
print(f'OK: {len(train_df)} filas, {len(train_df.columns)} columnas')
print(f'Columnas: {list(train_df.columns)}')

In [ ]:
# Revisar porcentaje de NaN por columna — un NaN alto indica que no se encontraron
# imágenes válidas para muchas muestras (p.ej. zona siempre nublada)
print('NaNs por columna (%):')
print((train_df.isna().mean() * 100).round(1).to_string())

# Verificar distribución de los índices calculados
print('\nDistribucion de indices:')
for col in ['NDTI', 'NDVI_wat', 'nir_blue', 'ndvi_land', 'ndbi', 'bsi']:
    if col in train_df.columns:
        d = train_df[col].dropna()
        print(f'  {col:12s}: mean={d.mean():.3f}  std={d.std():.3f}  '
              f'[{d.min():.3f}, {d.max():.3f}]  NaN={train_df[col].isna().mean()*100:.1f}%')

In [ ]:
# Subir el CSV al workspace de Snowflake para usarlo en el notebook de modelado
# PUT copia un archivo local al almacenamiento interno de Snowflake
train_df.to_csv(f'/tmp/{TRAIN_FINAL}', index=False)
session.sql(f"""
    PUT file:///tmp/{TRAIN_FINAL}
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()
print(f'{TRAIN_FINAL} subido')

## Extracción — validación

In [ ]:
# Cargar el template de validación (las filas para las que hay que predecir)
validation_df = pd.read_csv('submission_template.csv')
print(f'Registros de validacion: {len(validation_df)}')

In [ ]:
VAL_PARTIAL = 'landsat_v4_val_partial.csv'
VAL_FINAL   = 'landsat_features_validation_v4.csv'

if os.path.exists(VAL_FINAL) and len(pd.read_csv(VAL_FINAL)) == len(validation_df):
    print(f'Ya completo, saltando')
else:
    processed_rows = len(pd.read_csv(VAL_PARTIAL)) if os.path.exists(VAL_PARTIAL) else 0
    file_exists    = os.path.exists(VAL_PARTIAL)
    print(f'Reanudando desde fila {processed_rows}')

    for start in range(processed_rows, len(validation_df), 50):
        end   = min(start + 50, len(validation_df))
        batch = validation_df.iloc[start:end]
        print(f'Filas {start}-{end}')

        feats = batch.progress_apply(compute_landsat_values_v4, axis=1)
        feats['Latitude']    = batch['Latitude'].values
        feats['Longitude']   = batch['Longitude'].values
        feats['Sample Date'] = batch['Sample Date'].values

        feats.to_csv(VAL_PARTIAL, mode='a', header=not file_exists, index=False)
        file_exists = True

    df = pd.read_csv(VAL_PARTIAL)
    col_order = ['Latitude', 'Longitude', 'Sample Date',
                 'nir', 'green', 'red', 'blue', 'swir16', 'swir22',
                 'NDMI', 'MNDWI', 'NDTI', 'NDVI_wat', 'nir_blue',
                 'ndvi_land', 'ndbi', 'bsi']
    df = df[[c for c in col_order if c in df.columns]]
    df.to_csv(VAL_FINAL, index=False)
    if os.path.exists(VAL_PARTIAL):
        os.remove(VAL_PARTIAL)
    print(f'Extraccion completa: {len(df)} registros')

val_df = pd.read_csv(VAL_FINAL)
assert len(val_df) == len(validation_df), f'Desfase: {len(val_df)} vs {len(validation_df)}'
print(f'OK: {len(val_df)} filas')

In [ ]:
val_df.to_csv(f'/tmp/{VAL_FINAL}', index=False)
session.sql(f"""
    PUT file:///tmp/{VAL_FINAL}
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()
print(f'{VAL_FINAL} subido')